In [6]:
# 1. Montar Drive & caminhos
from google.colab import drive
import os, subprocess

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
        !rm -rf /content/drive/*
    drive.mount(MNT, force_remount=True)

ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else \
       ("MeuDrive" if os.path.exists("/content/drive/MeuDrive") else "MyDrive")
BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
PROC_PATH = os.path.join(BASE_PATH, "data_processed")
print("PROC_PATH:", PROC_PATH)

PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [8]:
# 2. Carregar dados processados
import pandas as pd, numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

df_ibov = pd.read_csv(os.path.join(PROC_PATH, "ibovespa_clean.csv"))
df_news = pd.read_csv(os.path.join(PROC_PATH, "noticias_clean.csv"))

# corrigir nomes das colunas
df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

In [10]:
# 3. Feature simples: sentimento proxy por comprimento (placeholder)
df_news["sent_len"] = df_news["clean_text"].fillna("").apply(lambda s: len(s.split()))
daily = df_news.groupby("data")["sent_len"].mean().reset_index()

In [11]:
# 4. Merge com IBOV e rótulo (direção D+1)
df = pd.merge(df_ibov.sort_values("data"), daily, on="data", how="left").fillna(0)
df["ret1"] = df["close"].pct_change().shift(-1)           # retorno do dia seguinte
df = df.dropna().copy()
df["y"] = (df["ret1"] > 0).astype(int)
X = df[["sent_len"]].values
y = df["y"].values

In [12]:
# 5. Validação temporal (walk-forward simples)
tscv = TimeSeriesSplit(n_splits=3)
aucs, mdas = [], []
for tr, te in tscv.split(X):
    clf = LogisticRegression(max_iter=2000).fit(X[tr], y[tr])
    proba = clf.predict_proba(X[te])[:,1]
    pred  = (proba > 0.5).astype(int)
    aucs.append(roc_auc_score(y[te], proba))
    mdas.append(accuracy_score(y[te], pred))

print({
    "1d": {"AUC": float(np.mean(aucs)), "MDA": float(np.mean(mdas))}
})

{'1d': {'AUC': 0.6111111111111112, 'MDA': 0.75}}
